# ClinicalIQ Asia — Databricks Data Load
Load structured clinical trial data + unstructured document text into Unity Catalog.

**Prerequisites:**
- CSV files uploaded to: `<YOUR_STORAGE_ACCOUNT>.blob.core.windows.net/data/clinical/csv_files/`
- Azure storage accessible from Databricks workspace
- Catalog: `<YOUR_CATALOG>`

In [ ]:
USE CATALOG `<YOUR_CATALOG>`;
CREATE SCHEMA IF NOT EXISTS clinical;
USE SCHEMA clinical;

In [ ]:
%python
base_path = "abfss://data@<YOUR_STORAGE_ACCOUNT>.dfs.core.windows.net/clinical/csv_files"

# Structured tables: CSV filename → table name
structured_tables = [
    ("sponsors.csv", "tbl_sponsor"),
    ("sites.csv", "tbl_site"),
    ("investigators.csv", "tbl_investigator"),
    ("drugs.csv", "tbl_drug"),
    ("trials.csv", "tbl_trial"),
    ("trial_arms.csv", "tbl_trial_arm"),
    ("enrollments.csv", "tbl_enrollment"),
    ("visits.csv", "tbl_visit"),
    ("lab_results.csv", "tbl_lab_result"),
    ("adverse_events.csv", "tbl_adverse_event"),
    ("conmeds.csv", "tbl_conmed"),
    ("vital_signs.csv", "tbl_vital_sign"),
    ("regulatory.csv", "tbl_regulatory_submission"),
    ("protocol_deviations.csv", "tbl_protocol_deviation"),
    ("milestones.csv", "tbl_milestone"),
    ("budgets.csv", "tbl_budget"),
]

for csv_file, table in structured_tables:
    file_path = f"{base_path}/{csv_file}"
    try:
        df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .option("nullValue", "None")
              .option("emptyValue", "")
              .csv(file_path))
        df.write.mode("overwrite").saveAsTable(f"`<YOUR_CATALOG>`.clinical.{table}")
        print(f"Created: {table} ({df.count():,} rows)")
    except Exception as e:
        print(f"Error: {table}: {e}")

In [ ]:
%python
# Load document metadata (3 meta CSVs → tbl_document)
doc_meta_files = [
    "clinicaltrials_docs_meta.csv",
    "pubmed_docs_meta.csv",
    "dailymed_docs_meta.csv",
]

first = True
for csv_file in doc_meta_files:
    file_path = f"{base_path}/{csv_file}"
    try:
        df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .option("nullValue", "None")
              .option("emptyValue", "")
              .csv(file_path))
        mode = "overwrite" if first else "append"
        df.write.mode(mode).saveAsTable(f"`<YOUR_CATALOG>`.clinical.tbl_document")
        first = False
        print(f"Loaded: {csv_file} ({df.count():,} rows)")
    except Exception as e:
        print(f"Error: {csv_file}: {e}")

In [ ]:
%python
# Populate parsed_text in tbl_document from the full-text CSVs
# NOTE: multiLine=true needed because parsed_text contains newlines
# NOTE: dropDuplicates needed because some CSVs produce duplicate doc_ids

# First add parsed_text column if not present
try:
    spark.sql("ALTER TABLE `<YOUR_CATALOG>`.clinical.tbl_document ADD COLUMNS (parsed_text STRING)")
except Exception:
    pass  # column already exists

doc_text_files = [
    "clinicaltrials_docs.csv",
    "pubmed_docs.csv",
    "dailymed_docs.csv",
]

for csv_file in doc_text_files:
    file_path = f"{base_path}/{csv_file}"
    try:
        df_text = (spark.read
                   .option("header", "true")
                   .option("inferSchema", "true")
                   .option("nullValue", "None")
                   .option("multiLine", "true")
                   .option("escape", '"')
                   .csv(file_path)
                   .select("doc_id", "parsed_text")
                   .dropDuplicates(["doc_id"]))
        df_text.createOrReplaceTempView("tmp_doc_text")
        spark.sql("""
            MERGE INTO `<YOUR_CATALOG>`.clinical.tbl_document t
            USING tmp_doc_text s ON t.doc_id = s.doc_id
            WHEN MATCHED THEN UPDATE SET t.parsed_text = s.parsed_text
        """)
        print(f"Updated parsed_text from {csv_file} ({df_text.count():,} rows)")
    except Exception as e:
        print(f"Error updating text from {csv_file}: {e}")


In [ ]:
-- Verify row counts
SELECT 'tbl_sponsor' AS tbl, COUNT(*) AS cnt FROM `<YOUR_CATALOG>`.clinical.tbl_sponsor
UNION ALL SELECT 'tbl_site', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_site
UNION ALL SELECT 'tbl_investigator', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_investigator
UNION ALL SELECT 'tbl_drug', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_drug
UNION ALL SELECT 'tbl_trial', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_trial
UNION ALL SELECT 'tbl_trial_arm', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_trial_arm
UNION ALL SELECT 'tbl_enrollment', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_enrollment
UNION ALL SELECT 'tbl_visit', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_visit
UNION ALL SELECT 'tbl_lab_result', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_lab_result
UNION ALL SELECT 'tbl_adverse_event', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_adverse_event
UNION ALL SELECT 'tbl_conmed', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_conmed
UNION ALL SELECT 'tbl_vital_sign', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_vital_sign
UNION ALL SELECT 'tbl_regulatory_submission', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_regulatory_submission
UNION ALL SELECT 'tbl_protocol_deviation', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_protocol_deviation
UNION ALL SELECT 'tbl_milestone', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_milestone
UNION ALL SELECT 'tbl_budget', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_budget
UNION ALL SELECT 'tbl_document', COUNT(*) FROM `<YOUR_CATALOG>`.clinical.tbl_document
ORDER BY tbl

In [ ]:
-- Verify document text was populated
SELECT doc_source, COUNT(*) AS total, COUNT(parsed_text) AS has_text,
       ROUND(AVG(LENGTH(parsed_text))) AS avg_text_len
FROM `<YOUR_CATALOG>`.clinical.tbl_document
GROUP BY doc_source
ORDER BY doc_source

In [ ]:
-- Sample: verify key structured data
SELECT trial_id, phase, status, therapeutic_area, actual_enrollment
FROM `<YOUR_CATALOG>`.clinical.tbl_trial
LIMIT 5